<div align="center">

# VLM-MASK-REASONER for VOID: Quadmask Generation Pipeline

**Automatic interaction-aware mask generation using SAM2, Gemini VLM, and LangSAM**

[Project Page](https://void-model.github.io/) | [Paper](https://arxiv.org/abs/2604.02296) | [HuggingFace](https://huggingface.co/netflix/void-model)

</div>

This notebook demonstrates the **VLM-MASK-REASONER** pipeline that generates quadmasks for VOID video inpainting. The pipeline:

1. **Stage 1:** Uses SAM2 to segment the primary object (what you want to remove)
2. **Stage 2:** Uses Gemini VLM to identify affected objects (shadows, reflections, interactions)
3. **Stage 3:** Uses LangSAM to segment affected objects
4. **Stage 4:** Combines everything into a quadmask with 4 semantic values (0, 63, 127, 255)

**Requirements:** 
- GPU runtime (T4 or better recommended)
- Gemini API key (free at https://aistudio.google.com/app/apikey)

**Note:** This notebook uses **LangSAM** for Stage 3 instead of SAM3 for Python 3.10 compatibility on Colab.

## 1. Setup & Installation

Install all required dependencies including SAM2, LangSAM, and download checkpoints.

In [ ]:
# Clone the repo (skip if already cloned)
!git clone git@github.com:tharencandi/void-model.git 2>/dev/null || echo "Repo already exists"
!git checkout feat/stage3-segmentation-model-flag
%cd /content/void-model

In [ ]:
# Install main requirements
!pip install -q opencv-python-headless numpy pillow tqdm google-generativeai

# Install SAM2 for Stage 1
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

# Install LangSAM for Stage 3 (Python 3.10 compatible)
!pip install -q -U git+https://github.com/luca-medeiros/lang-segment-anything.git

print("✓ All dependencies installed")

## 2. Download SAM2 Checkpoint

In [ ]:
from pathlib import Path

sam2_checkpoint = Path("sam2_hiera_large.pt")

if not sam2_checkpoint.exists():
    print("Downloading SAM2 checkpoint (2.4GB)...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
    print("✓ SAM2 checkpoint downloaded")
else:
    print("✓ SAM2 checkpoint already exists")

## 3. Set Gemini API Key

Get API key at: https://aistudio.google.com/app/apikey

In [ ]:
import os
from google.colab import userdata

try:
    # Get your Gemini API key from Colab secrets
    gemini_api_key = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = gemini_api_key
    print("✓ Gemini API key loaded from Colab secrets.")
except userdata.SecretNotFoundError:
    print("❌ GEMINI_API_KEY not found in Colab secrets. Please add it to Colab secrets or set it manually.")
except Exception as e:
    print(f"An error occurred: {e}")


## 4. Prepare Sample Video & Config

We'll use the "lime" sample - a glass being removed, causing the lime to fall.

The pipeline requires a `config_points.json` file with point coordinates. You have **two options**:

**Option A:** Use the interactive Point Selector GUI locally (recommended for precision)  
**Option B:** Manually create the config (good for quick testing)

---

### Understanding config_points.json Format

The config file specifies:
- **video_path**: Path to your input video
- **output_dir**: Where to save generated masks
- **instruction**: Text description of what to remove
- **primary_points**: Click coordinates on the object to segment (for SAM2)
  - `frame_idx`: Which frame to click on (0 = first frame)
  - `points`: List of [x, y] coordinates (click on the object)
  - `labels`: 1 for positive click (include), 0 for negative (exclude)
- **min_grid**: Grid size for gridification (8 is standard)

**Example config_points.json:**
```json
{
  "videos": [
    {
      "video_path": "/path/to/video.mp4",
      "output_dir": "/path/to/output",
      "instruction": "remove the glass",
      "primary_points": [
        {
          "frame_idx": 0,
          "points": [[336, 240]],
          "labels": [1]
        }
      ],
      "min_grid": 8
    }
  ]
}
```

You can add multiple videos to the `videos` array, and add points on multiple frames if the object moves significantly.

### Option A: Use Point Selector GUI Locally (Recommended)

For precise point selection, you can run the interactive GUI tool locally on your machine:

**Step 1:** Create a simple config file (without points):
```json
{
  "videos": [
    {
      "video_path": "/path/to/your/video.mp4",
      "output_dir": "/path/to/output",
      "instruction": "remove the glass"
    }
  ]
}
```

**Step 2:** Run the point selector GUI:
```bash
python VLM-MASK-REASONER/point_selector_gui.py --config my_config.json
```

**Step 3:** Use the GUI to:
- Navigate frames with the slider or arrow keys
- Click on the object you want to remove (multiple clicks for better segmentation)
- Add points on multiple frames if the object moves significantly
- Press "Save All Points" or Space to save

This creates `my_config_points.json` with exact coordinates.

**Step 4:** Upload the `_points.json` file to Colab and use it in the pipeline.

---

### Option B: Manual Config Creation (Quick Start)

---

### Demo: Using Manual Config Creation (Option B)

For this demo, we'll manually create the config. The code below creates a working config for the lime sample with pre-selected coordinates.

In [ ]:
import json
from pathlib import Path

# Create config for the lime sample video with actual coordinates
# To use your own video: replace video_path, adjust points coordinates
config_data = {
    "videos": [
        {
            "video_path": "sample/lime/input_video.mp4",      # Path to video
            "output_dir": "sample/lime/masks_output",          # Output directory
            "instruction": "remove the glass",                  # What to remove
            "primary_points_by_frame": {
                "0": [
                    [2150, 1243],
                    [2702, 1276],
                    [2452, 1886],
                    [2448, 1507],
                    [2424, 1104]
                ]
            }, # Frame 0: click on glass at [x, y]
            "primary_frames": [
                0
            ],           # Frames with points
            "first_appears_frame": 0,        # First frame where object appears
            "primary_points": [
                [2150, 1243],
                [2702, 1276],
                [2452, 1886],
                [2448, 1507],
                [2424, 1104]
            ], # Flattened points (for backward compatibility)
            "min_grid": 8                    # Grid size for VLM analysis
        }
    ]
}

# Save config file
config_path = Path("config_points.json")
with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=2)

print("\n💡 For precise point selection, use the GUI tool from Option A!")

output_dir = Path(config_data["videos"][0]["output_dir"])print("   5. For multi-frame: '0': [[x1,y1]], '30': [[x2,y2]]")

output_dir.mkdir(parents=True, exist_ok=True)print("   4. Add more points: '0': [[x1,y1], [x2,y2], ...]")

print("   3. Update coordinates in 'primary_points_by_frame'")

print(f"✓ Config created: {config_path}")print("   2. Edit config_data above - update video_path and output_dir")

print(f"✓ Output directory: {output_dir}")print("   1. Upload your video file to Colab")

print(f"✓ Using point coordinates: {config_data['videos'][0]['primary_points']}")print("\n📝 To use your own video:")

## 5. Run the Pipeline

You have **two options** to run the 4-stage pipeline:

**Option A:** Run all stages at once with `run_pipeline.sh` (recommended - single command)  
**Option B:** Run each stage individually (good for debugging or customization)

---

### Option A: Run All Stages at Once (Recommended)

In [ ]:
# Run the complete pipeline with a single command
# This runs all 4 stages automatically: SAM2 → Gemini VLM → LangSAM → Combine

!bash VLM-MASK-REASONER/run_pipeline.sh config_points.json \
    --sam2-checkpoint sam2_hiera_large.pt \
    --stage3-segmentation-model langsam \
    --device cuda

print("\n" + "=" * 70)
print("✓ Complete Pipeline Finished!")
print("=" * 70)
print("\nGenerated files in output directory:")
print("  - black_mask.mp4 (Stage 1)")
print("  - vlm_analysis.json (Stage 2)")
print("  - grey_mask.mp4 (Stage 3)")
print("  - quadmask_0.mp4 (Stage 4)")
print("\nContinue to Section 6 to visualize results.")

---

### Option B: Run Stages Individually

Run each stage separately if you want to inspect intermediate results or debug issues.

**Stage 1:** SAM2 Segmentation (primary object)  
**Stage 2:** Gemini VLM Analysis (identify affected objects)  
**Stage 3:** LangSAM Segmentation (affected objects)  
**Stage 4:** Combine into Quadmask

In [ ]:
# Run Stage 1: SAM2 Segmentation
print("=" * 70)
print("Stage 1: SAM2 Segmentation")
print("=" * 70)

!python VLM-MASK-REASONER/stage1_sam2_segmentation.py \
    --config config_points.json \
    --sam2-checkpoint sam2_hiera_large.pt \
    --device cuda

print("\n✓ Stage 1 complete: black_mask.mp4 generated")

In [ ]:
# Run Stage 2: VLM Analysis with Gemini
print("\n" + "=" * 70)
print("Stage 2: VLM Analysis (Gemini)")
print("=" * 70)

!python VLM-MASK-REASONER/stage2_vlm_analysis.py \
    --config config_points.json

print("\n✓ Stage 2 complete: vlm_analysis.json generated")

In [ ]:
# Run Stage 3: Grey Mask Generation with LangSAM
print("\n" + "=" * 70)
print("Stage 3: Generate Grey Masks (LangSAM)")
print("=" * 70)

!python VLM-MASK-REASONER/stage3a_generate_grey_masks_v2.py \
    --config config_points.json \
    --segmentation-model langsam

print("\n✓ Stage 3 complete: grey_mask.mp4 generated")

In [ ]:
# Run Stage 4: Combine into Quadmask
print("\n" + "=" * 70)
print("Stage 4: Combine Masks into Quadmask")
print("=" * 70)

!python VLM-MASK-REASONER/stage4_combine_masks.py \
    --config config_points.json

print("\n✓ Stage 4 complete: quadmask_0.mp4 generated")
print("\n" + "=" * 70)
print("Pipeline Complete!")
print("=" * 70)

## 6. Visualize Results

Display the generated masks and VLM analysis to verify the pipeline output.

In [ ]:
import cv2
import numpy as np
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Read first frame of each video
def read_first_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

# Load frames
input_frame = read_first_frame(output_dir / "input_video.mp4")
black_mask = read_first_frame(output_dir / "black_mask.mp4")
grey_mask = read_first_frame(output_dir / "grey_mask.mp4")
quadmask = read_first_frame(output_dir / "quadmask_0.mp4")

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0, 0].imshow(input_frame)
axes[0, 0].set_title("Input Video (Frame 0)", fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(black_mask, cmap='gray')
axes[0, 1].set_title("Black Mask (Primary Object)\n0=remove, 255=keep", fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(grey_mask, cmap='gray')
axes[1, 0].set_title("Grey Mask (Affected Objects)\n127=affected, 255=background", fontsize=14, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(quadmask)
axes[1, 1].set_title("Final Quadmask\n0=primary, 63=overlap, 127=affected, 255=bg", fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(output_dir / "pipeline_results.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to:", output_dir / "pipeline_results.png")

In [ ]:
# Show VLM analysis results
import json

vlm_analysis_path = output_dir / "vlm_analysis.json"
if vlm_analysis_path.exists():
    print("=" * 70)
    print("Gemini VLM Analysis Results")
    print("=" * 70)
    
    with open(vlm_analysis_path, 'r') as f:
        analysis = json.load(f)
    
    print(f"\nPrimary Object: {analysis.get('primary_object', 'N/A')}")
    print(f"Scene Description: {analysis.get('scene_description', 'N/A')}")
    
    affected = analysis.get('affected_objects', [])
    print(f"\nAffected Objects ({len(affected)}):")
    for i, obj in enumerate(affected, 1):
        print(f"  {i}. {obj.get('noun', 'N/A')} - {obj.get('reasoning', 'N/A')}")
    
    print("\n" + "=" * 70)

## 7. Download Results

Download the quadmask and use it with VOID Pass 1 inference.

**Quadmask Values:**
- **0 (black):** Primary object to remove
- **63:** Overlap region
- **127 (grey):** Affected objects
- **255 (white):** Background

**Try other samples:** Change coordinates to `"moving_ball"` or `"pillow"` sample videos

**Use your own video:** Upload a video, modify the config with correct path and click points, re-run from Section 5

In [ ]:
from google.colab import files

# Download the quadmask
quadmask_path = output_dir / "quadmask_0.mp4"
if quadmask_path.exists():
    print("Downloading quadmask_0.mp4...")
    files.download(str(quadmask_path))
    print("✓ Download complete!")
else:
    print("❌ Quadmask not found. Check pipeline output above for errors.")